In [7]:
import os
from dotenv import load_dotenv
from tavily import TavilyClient
from groq import Groq

load_dotenv()
# --- Keys ---
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

if not GROQ_API_KEY:
    raise RuntimeError("Missing GROQ_API_KEY in .env")

if not TAVILY_API_KEY:
    raise RuntimeError("Missing TAVILY_API_KEY in .env")

# --- Clients ---
groq_client = Groq(api_key=GROQ_API_KEY)
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

def internet_search(query: str, max_results: int = 5, topic: str = "general") -> dict:
    """Run Tavily search."""
    return tavily_client.search(
        query=query,
        max_results=max_results,
        include_raw_content=False,
        topic=topic,
    )

def build_context(search_results: dict) -> str:
    """Convert Tavily results into compact context for the LLM."""
    results = search_results.get("results", [])
    if not results:
        return "No search results found."

    chunks = []
    for i, r in enumerate(results, start=1):
        title = r.get("title", "No title")
        url = r.get("url", "No URL")
        content = r.get("content", "No content")
        chunks.append(
            f"[Source {i}]\n"
            f"Title: {title}\n"
            f"URL: {url}\n"
            f"Content: {content}\n"
        )

    return "\n".join(chunks)

def ask_groq(question: str, context: str) -> str:
    """Send retrieved context plus user question to Groq."""
    prompt = f"""
You are an expert research assistant.

Answer the user's question using the context below.
Be clear, accurate, and concise.
If the context is insufficient, say so.

Context:
{context}

Question:
{question}
""".strip()

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are a helpful research assistant."},
            {"role": "user", "content": prompt},
        ],
        temperature=0,
    )

    return response.choices[0].message.content

def research_agent(query: str, topic: str = "general", max_results: int = 5):
    search_results = internet_search(query=query, max_results=max_results, topic=topic)
    context = build_context(search_results)
    answer = ask_groq(query, context)
    return answer, search_results

if __name__ == "__main__":
    query = "Summarize the top finance news for march top 5 articles from biggest magazines BBC Economist,Wallstreet journal, BusinessReview,Harvard, only read 5 articles,"
    answer, search_results = research_agent(query, topic="general", max_results=5)

    print("\n=== ANSWER ===\n")
    print(answer)

    print("\n=== SOURCES ===\n")
    for i, r in enumerate(search_results.get("results", []), start=1):
        print(f"{i}. {r.get('title')} - {r.get('url')}")


=== ANSWER ===

Based on the provided context, I can only summarize the top finance news from the Wall Street Journal (WSJ) as the other sources (BBC Economist, Business Review, and Harvard) are not directly mentioned in the context. Here's a summary of the top finance news from the WSJ:

1. **Wall Street Is Making Bullish Bets on the Economy** (Source 5): The article reports that Wall Street is making optimistic bets on the economy, with signs of economic optimism including rallying retail stocks and stubbornly high bond yields.
2. **March 2026 News Archive** (Source 1): The WSJ's digital archive of news articles and top headlines from March 2026 is available, but the specific articles are not mentioned in the context.
3. **The Wall Street Journal's News Archive for March 10, 2026** (Source 2): The WSJ's digital archive of news articles and top headlines from March 10, 2026, is available, but the specific articles are not mentioned in the context.
4. **The Wall Street Journal's News 

In [21]:
!pip install sentence-transformers

from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

  Using cached sentence_transformers-5.3.0-py3-none-any.whl.metadata (16 kB)
  Using cached transformers-5.3.0-py3-none-any.whl.metadata (32 kB)
  Using cached torch-2.10.0-cp314-cp314-win_amd64.whl.metadata (31 kB)
  Using cached scikit_learn-1.8.0-cp314-cp314-win_amd64.whl.metadata (11 kB)
  Using cached scipy-1.17.1-cp314-cp314-win_amd64.whl.metadata (60 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.3-cp314-cp314-win_amd64.whl.metadata (2.8 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached sentence_transforme

   -------------------------- ------------- 24.4/37.3 MB 78.9 kB/s eta 0:02:44
   -------------------------- ------------- 24.6/37.3 MB 79.9 kB/s eta 0:02:39
   -------------------------- ------------- 24.6/37.3 MB 79.9 kB/s eta 0:02:39
   -------------------------- ------------- 24.6/37.3 MB 79.9 kB/s eta 0:02:39
   -------------------------- ------------- 24.6/37.3 MB 79.9 kB/s eta 0:02:39
   -------------------------- ------------- 24.6/37.3 MB 79.9 kB/s eta 0:02:39
   -------------------------- ------------- 24.6/37.3 MB 79.9 kB/s eta 0:02:39
   -------------------------- ------------- 24.6/37.3 MB 79.9 kB/s eta 0:02:39
   -------------------------- ------------- 24.6/37.3 MB 79.9 kB/s eta 0:02:39
   -------------------------- ------------- 24.6/37.3 MB 79.9 kB/s eta 0:02:39
   -------------------------- ------------- 24.6/37.3 MB 79.9 kB/s eta 0:02:39
   -------------------------- ------------- 24.6/37.3 MB 79.9 kB/s eta 0:02:39
   -------------------------- ------------- 24.9/37.

C:\Users\Thato Bilankulu\OneDrive - Linkfields innovations\Documents\econ research test\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Thato Bilankulu\OneDrive - Linkfields innovations\Documents\econ research test\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Thato Bilankulu\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitat

KeyboardInterrupt: 

In [4]:
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

client = QdrantClient(":memory:")

vector_size = len(embeddings.embed_query("sample text"))

if not client.collection_exists("test"):
    client.create_collection(
        collection_name="test",
        vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE)
    )
vector_store = QdrantVectorStore(
    client=client,
    collection_name="test",
    embedding=embeddings,
)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)

print(f"Split blog post into {len(all_splits)} sub-documents.")

In [13]:
#from langchain_community.document_loaders import WebBaseLoader
#!pip install -qU langchain-qdrant
!pip install -qU langchain-groq

In [12]:
import getpass
import os

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

In [14]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
    # other params...
)

In [15]:
!pip install -qU langchain-huggingface

In [1]:
!pip install sentence-transformers
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-small-en-v1.5")

embedding = model.encode("Inflation impacts equity markets")

C:\Users\Thato Bilankulu\OneDrive - Linkfields innovations\Documents\econ research test\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Thato Bilankulu\OneDrive - Linkfields innovations\Documents\econ research test\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Thato Bilankulu\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support 

In [ ]:
!pip install hugging

In [1]:
from qdrant_client.models import Distance, VectorParams
from qdrant_client import QdrantClient
from langchain_qdrant import QdrantVectorStore
from langchain_core.documents import Document
from sentence_transformers import SentenceTransformer

# Embedding model
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

# Qdrant client
client = QdrantClient(":memory:")

vector_size = len(embeddings.embed_query("sample text"))

if not client.collection_exists("finance_articles"):
    client.create_collection(
        collection_name="finance_articles",
        vectors_config=VectorParams(
            size=vector_size,
            distance=Distance.COSINE
        )
    )

vector_store = QdrantVectorStore(
    client=client,
    collection_name="finance_articles",
    embedding=embeddings,
)

# Add documents
docs = [
    Document(page_content="S&P 500 rose after inflation cooled"),
    Document(page_content="Oil prices surged due to geopolitical tensions"),
]

vector_store.add_documents(docs)

# Query
results = vector_store.similarity_search("Why did the S&P 500 rise?")

for r in results:
    print(r.page_content)

C:\Users\Thato Bilankulu\OneDrive - Linkfields innovations\Documents\econ research test\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
C:\Users\Thato Bilankulu\OneDrive - Linkfields innovations\Documents\econ research test\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4948.99it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


S&P 500 rose after inflation cooled
Oil prices surged due to geopolitical tensions


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)

print(f"Split blog post into {len(all_splits)} sub-documents.")

Split blog post into 2 sub-documents.


In [3]:
document_ids = vector_store.add_documents(documents=all_splits)

print(document_ids[:3])

['c49b26a905d74d5fa2f93665dd3a5eeb', '6e24566951d946d4ade7f87434191cd9']
